# Demo: generate ops-app KPI gzip shard -> `raw_data/kpis/`

Reads random rows from `/Volumes/.../raw_data/310.csv.gz` and writes a uniquely named KPI shard into the Auto Loader incoming folder for `ops_app_bronze_tower_hourly_kpis`.

Output schema (headerless CSV):
1. `tower_id`
2. `event_ts`
3. `prb_utilization_pct`
4. `latency_ms`
5. `packet_loss_pct`
6. `throughput_mbps`

In [ ]:
SOURCE_GZIP = "/Volumes/cmegdemos_catalog/network_analytics_enablement/raw_data/310.csv.gz"
KPI_DIR = "/Volumes/cmegdemos_catalog/network_analytics_enablement/raw_data/kpis"
CATALOG = "cmegdemos_catalog"
SCHEMA = "network_analytics_enablement"

MIN_SAMPLE_ROWS = 8_000
MAX_SAMPLE_ROWS = 40_000

In [ ]:
import os
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd

assert os.path.exists(SOURCE_GZIP), f"Missing source file: {SOURCE_GZIP}"
os.makedirs(KPI_DIR, exist_ok=True)

tower_pdf = spark.sql(
    f"SELECT tower_id FROM {CATALOG}.{SCHEMA}.silver_tmobile_towers_seattle"
).toPandas()
assert not tower_pdf.empty, "No rows found in silver_tmobile_towers_seattle"
tower_ids = tower_pdf["tower_id"].astype("int64").to_numpy()

seed_df = pd.read_csv(SOURCE_GZIP, header=None, compression="gzip", dtype=str, low_memory=False)
seed_n = len(seed_df)
rng = np.random.default_rng()
hi = min(MAX_SAMPLE_ROWS, seed_n)
lo = min(MIN_SAMPLE_ROWS, hi)
target_n = int(rng.integers(lo, hi + 1)) if hi >= lo else seed_n
target_n = max(1, min(target_n, seed_n))
sampled_seed = seed_df.sample(n=target_n, replace=False, random_state=rng)

now = datetime.now(timezone.utc)
hours_back = rng.integers(0, 14 * 24, size=target_n)
event_ts = [
    (now - pd.Timedelta(hours=int(h))).replace(minute=0, second=0, microsecond=0)
    for h in hours_back
]

base_signal = pd.to_numeric(sampled_seed[13], errors="coerce").fillna(-95.0)
util = np.clip(35 + (rng.random(target_n) * 70) - (base_signal + 95) * 0.2, 5, 100)
latency = np.clip(30 + util * 1.1 + rng.normal(0, 8, target_n), 10, 220)
packet_loss = np.clip(0.2 + util * 0.03 + rng.normal(0, 0.5, target_n), 0, 18)
throughput = np.clip(240 - util * 1.6 + rng.normal(0, 12, target_n), 20, 300)

out_pdf = pd.DataFrame(
    {
        "tower_id": rng.choice(tower_ids, size=target_n, replace=True),
        "event_ts": pd.to_datetime(event_ts).strftime("%Y-%m-%d %H:%M:%S"),
        "prb_utilization_pct": np.round(util, 2),
        "latency_ms": np.round(latency, 2),
        "packet_loss_pct": np.round(packet_loss, 3),
        "throughput_mbps": np.round(throughput, 2),
    }
)

ts = now.strftime("%Y%m%dT%H%M%SZ")
suffix = uuid.uuid4().hex[:10]
out_name = f"ops_kpis_shard_{ts}_{suffix}.csv.gz"
out_path = os.path.join(KPI_DIR, out_name)
out_pdf.to_csv(out_path, header=False, index=False, compression="gzip")

print(f"Seed rows sampled from 310.csv.gz: {target_n:,}")
print(f"Wrote KPI shard: {out_path}")
print("Next: run ops_app_network_analytics_pipeline")